# Forsterite dissolution

Reads the `<case>_erw` `met_forcing_rxn` runs on s3 and builds a forsterite dissolution zarr store
to save at `s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations/postprocessed/met_forcing_rxn/`:

`forsterite_dissolution.zarr`, dims `(site 8, case 4, time 3650, year 10)`.

| variable | dims | what it is |
|---|---|---|
| `dissolved` | `(site, case, time)` | cumulative mass of forsterite dissolved since day 0 |
| `dissolved_diff` | `(site, case, time)` | `dissolved` minus `dissolved` of the hourly case |
| `dissolved_pcterr` | `(site, case, time)` | `dissolved_diff` as a % of the hourly cumulative |
| `annual_dissolution` | `(site, case, year)` | mass dissolved within each 365-day year |
| `annual_dissolution_diff` | `(site, case, year)` | that year's dissolution, minus the hourly case |
| `annual_dissolution_pcterr` | `(site, case, year)` | that year's dissolution as a % error vs hourly |
| `inventory0` | `(site)` | forsterite present at day 0 |

Source data from domain-integrated mineral inventory `<case>_o.mms`, column `forst-ph`; dissolved
mass is `(M(0) - M(t)) x 140.6931 g/mol`. 

## 0. Setup

In [2]:
import io
import re
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import s3fs
import xarray as xr

from byte_util.util import all_sites, states_per_site

# ----------------------------------------------------------------- knobs
WRITE = True          # False builds everything in memory but writes nothing to S3
OVERWRITE = True      # mode="w" on the zarr store; False refuses to clobber
WORKERS = 8           # parallel S3 reads
# ------------------------------------------------------------------------

SITES = list(all_sites)                              # all eight
CASES = ["hourly", "daily", "monthly", "longterm"]   # ordered coarse-ward
REFERENCE = "hourly"                                 # every difference is against this case
TREAT = "erw"                                        # only the amended runs carry forsterite

RUN_DAYS = 3650
YEAR_DAYS = 365                                      # MIN3P's calendar here: no leap days
N_YEARS = RUN_DAYS // YEAR_DAYS

MINERAL = "forst-ph"
MOLAR_MASS = 140.6931        # g/mol, from `simulations/databases/mineral.dbs`
QUANTUM_MOL = 1e-4           # `_o.mms` writes 5 significant figures on a ~7 mol inventory -- for calculating lower reporting bound

BUCKET_ROOT = ("carbonplan-carbon-removal/ew-workflows-data/min3p/simulations"
               "/met_forcing_rxn/min3p_runs")
OUT_ROOT = ("s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations"
            "/postprocessed/met_forcing_rxn")

RUNS = [(s, c) for s in SITES for c in CASES]
fs = s3fs.S3FileSystem()

print(f"{len(RUNS)} runs: {len(SITES)} sites x {len(CASES)} cases, {TREAT} only")

32 runs: 8 sites x 4 cases, erw only


## 1. The time axes

`time` is the end of each day, in elapsed days; `year` is the year index. 

In [3]:
day_ends = np.arange(1, RUN_DAYS + 1, dtype=float)
years = np.arange(1, N_YEARS + 1)
year_ends = years * float(YEAR_DAYS)
year_index = np.searchsorted(day_ends, year_ends)      # where each year closes in `time`

assert np.array_equal(day_ends[year_index], year_ends), "years do not fall on day ends"
print(f"{day_ends.size} days ending {day_ends[:3]} ... {day_ends[-2:]} d")
print(f"{years.size} years ending {year_ends[:3]} ... {year_ends[-2:]} d "
      f"at time index {year_index[:3]} ... {year_index[-2:]}")

3650 days ending [1. 2. 3.] ... [3649. 3650.] d
10 years ending [ 365.  730. 1095.] ... [3285. 3650.] d at time index [ 364  729 1094] ... [3284 3649]


## 2. Read each run

Two objects per run, both read as bytes over the network: `<case>_o.mms` (the domain mineral
inventory) and the tail of `<case>_o.gen` (the exit banner).

In [4]:
def run_key(site, case):
    return f"{BUCKET_ROOT}/{site}/{case}_{TREAT}"


def read_transient_bytes(raw, path=""):
    """A MIN3P transient output file, given its bytes, as a DataFrame.
    """
    lines = raw.decode("latin1").splitlines()
    zone = next((i for i, ln in enumerate(lines) if ln.lstrip().lower().startswith("zone")), None)
    if zone is None:
        raise ValueError(f"no zone line in {path}")
    v0 = next(i for i, ln in enumerate(lines) if ln.lstrip().lower().startswith("variables"))
    names = [n.strip() for n in re.findall(r'"([^"]*)"', "\n".join(lines[v0:zone]))]
    df = pd.read_csv(io.StringIO("\n".join(lines[zone + 1:])), sep=r"\s+",
                     header=None, engine="c")
    if df.shape[1] != len(names):
        raise ValueError(f"{path}: {len(names)} variable names, {df.shape[1]} columns")
    df.columns = names
    return df


def exit_status(site, case):
    """The last banner MIN3P wrote, from a range read of the tail of `<case>_o.gen`."""
    text = fs.cat_file(f"{run_key(site, case)}/{case}_o.gen", start=-2000).decode("latin1")
    if "normal exit" in text:
        return "normal exit"
    fail = re.search(r"\*+\s*(.*?exit.*?|failure.*?)\s*\*+", text, re.I)
    return fail.group(1).strip() if fail else "unknown"


def read_inventory(site, case):
    """`(time in days, forsterite moles in the domain)` for one run, de-duplicated."""
    path = f"{run_key(site, case)}/{case}_o.mms"
    df = read_transient_bytes(fs.cat_file(path), path)
    t = df["time"].to_numpy(dtype=float)
    m = df[MINERAL].to_numpy(dtype=float)
    # MIN3P repeats a timestamp when it lands on a forced output time; keep the last of each.
    keep = np.ones(t.size, dtype=bool)
    keep[:-1] = t[1:] != t[:-1]
    t, m = t[keep], m[keep]
    if t[0] != 0.0:
        raise ValueError(f"{path} does not start at t = 0")
    if t[-1] < RUN_DAYS - 1e-6:
        raise ValueError(f"{path} stops at day {t[-1]}, not {RUN_DAYS}")
    return t, m


with ThreadPoolExecutor(WORKERS) as pool:
    statuses = dict(zip(RUNS, pool.map(lambda r: exit_status(*r), RUNS)))
bad = {r: s for r, s in statuses.items() if s != "normal exit"}
assert not bad, f"{len(bad)} runs did not exit normally: {bad}"
print(f"all {len(RUNS)} runs exited normally")

with ThreadPoolExecutor(WORKERS) as pool:
    records = dict(zip(RUNS, pool.map(lambda r: read_inventory(*r), RUNS)))
print(f"read {len(records)} inventories; rows per run "
      f"{min(len(t) for t, _ in records.values())}-{max(len(t) for t, _ in records.values())}")

all 32 runs exited normally
read 32 inventories; rows per run 3674-51745


## 3. Cumulative dissolved mass 

In [5]:
inventory0 = {}
for site in SITES:
    m0 = {case: records[(site, case)][1][0] for case in CASES}
    assert len(set(m0.values())) == 1, f"{site}: cases disagree on the initial inventory, {m0}"
    inventory0[site] = m0[REFERENCE]
    assert inventory0[site] > 0, f"{site} has no forsterite at day 0"

for site in SITES:
    mol = inventory0[site]
    print(f"{site:14s} {mol:7.4f} mol   {mol * MOLAR_MASS / 1000:7.4f} kg   "
          f"{mol * MOLAR_MASS / 100:7.3f} t/ha")

Cecil           6.9893 mol    0.9833 kg     9.833 t/ha
Flanagan        6.9893 mol    0.9833 kg     9.833 t/ha
HoustonBlack    6.9893 mol    0.9833 kg     9.833 t/ha
Kalamazoo       6.9893 mol    0.9833 kg     9.833 t/ha
Kuma            6.9893 mol    0.9833 kg     9.833 t/ha
Palouse         6.9893 mol    0.9833 kg     9.833 t/ha
Pullman         6.9893 mol    0.9833 kg     9.833 t/ha
Yolo            6.9893 mol    0.9833 kg     9.833 t/ha


In [6]:
slip = {}       # per run, the largest backward step in the raw inventory, in mol


def cumulative(site, case, at):
    """Mass of forsterite dissolved by each instant in `at`, in kg."""
    t, m = records[(site, case)]
    drop = np.maximum.accumulate(m[0] - m)      # kill any jitter of a flat record
    slip[(site, case)] = float(np.max(drop - (m[0] - m)))
    assert slip[(site, case)] <= 2.5 * QUANTUM_MOL, (
        f"{site}/{case} forsterite re-precipitates by {slip[(site, case)]:.2e} mol")
    return np.interp(at, t, drop) * MOLAR_MASS / 1000.0


dissolved = np.full((len(SITES), len(CASES), day_ends.size), np.nan)
for i, site in enumerate(SITES):
    for j, case in enumerate(CASES):
        dissolved[i, j] = cumulative(site, case, day_ends)
    print(f"  {site:14s} done", flush=True)

# The years close on day ends, so the annual view is a slice of the daily one rather than a second
# interpolation: the cumulative at each year end, differenced. The curve starts at zero, so the
# first year is its own value.
annual_dissolution = np.diff(dissolved[..., year_index], prepend=0.0, axis=-1)

assert (annual_dissolution >= -1e-12).all(), "a year dissolved a negative mass"
worst = max(slip, key=slip.get)
print(f"\nlargest backward step in any raw inventory: {slip[worst]:.1e} mol "
      f"({worst[0]}/{worst[1]}), against a {QUANTUM_MOL:.0e} mol output quantum")
print(f"\ncumulative dissolved at day {RUN_DAYS:.0f} (kg):")
print(pd.DataFrame(dissolved[..., -1], index=SITES, columns=CASES).round(4))

  Cecil          done
  Flanagan       done
  HoustonBlack   done
  Kalamazoo      done
  Kuma           done
  Palouse        done
  Pullman        done
  Yolo           done

largest backward step in any raw inventory: 0.0e+00 mol (Cecil/hourly), against a 1e-04 mol output quantum

cumulative dissolved at day 3650 (kg):
              hourly   daily  monthly  longterm
Cecil         0.9288  0.9293   0.9390    0.9448
Flanagan      0.9321  0.9324   0.9437    0.9481
HoustonBlack  0.7528  0.7591   0.8095    0.8383
Kalamazoo     0.9732  0.9735   0.9808    0.9824
Kuma          0.5693  0.5699   0.5875    0.6413
Palouse       0.8438  0.8412   0.8563    0.8671
Pullman       0.4535  0.4521   0.4434    0.4273
Yolo          0.7425  0.7420   0.7778    0.7976


## 4. Differences and percent errors against the hourly case

The one masked case is `dissolved_pcterr` in the opening days, where the hourly cumulative is still
only a few output quanta and a percentage of it means nothing. The threshold is set at `FLOOR`. 

In [9]:
REF = CASES.index(REFERENCE)
FLOOR = 100 * QUANTUM_MOL * MOLAR_MASS / 1000.0        # kg; below this, no percentage is reported
print(f"percent-error floor: {FLOOR:.2e} kg ({100 * QUANTUM_MOL:.0e} mol)")


def pct_error(diff, reference):
    """100 x diff / reference, NaN where the reference is too small to carry a percentage."""
    out = np.full(diff.shape, np.nan)
    ok = np.broadcast_to(np.abs(reference) > FLOOR, diff.shape)
    np.divide(100.0 * diff, reference, out=out, where=ok)
    return out


dissolved_diff = dissolved - dissolved[:, [REF]]
annual_dissolution_diff = annual_dissolution - annual_dissolution[:, [REF]]

dissolved_pcterr = pct_error(dissolved_diff, dissolved[:, [REF]])
annual_dissolution_pcterr = pct_error(annual_dissolution_diff, annual_dissolution[:, [REF]])

for name, arr in [("dissolved_pcterr", dissolved_pcterr),
                  ("annual_dissolution_pcterr", annual_dissolution_pcterr)]:
    ref = arr[:, REF]
    assert np.all((ref == 0) | np.isnan(ref)), f"{name} is not zero for the reference case"
    masked = int(np.isnan(arr).sum())
    print(f"{name:28s} {np.nanmin(arr):+8.2f} to {np.nanmax(arr):+8.2f} %   "
          f"{masked} of {arr.size} masked")

print("\n% error on the 10-year cumulative:")
print(pd.DataFrame(dissolved_pcterr[..., -1], index=SITES, columns=CASES).round(2))

percent-error floor: 1.41e-03 kg (1e-02 mol)
dissolved_pcterr                -8.58 to   +19.67 %   64 of 116800 masked
annual_dissolution_pcterr      -49.05 to   +19.35 %   0 of 320 masked

% error on the 10-year cumulative:
              hourly  daily  monthly  longterm
Cecil            0.0   0.05     1.09      1.72
Flanagan         0.0   0.03     1.24      1.71
HoustonBlack     0.0   0.83     7.53     11.36
Kalamazoo        0.0   0.03     0.78      0.94
Kuma             0.0   0.10     3.18     12.64
Palouse          0.0  -0.31     1.48      2.76
Pullman          0.0  -0.30    -2.22     -5.76
Yolo             0.0  -0.06     4.76      7.43


## 5. Assemble

In [10]:
tdim = ("site", "case", "time")
ydim = ("site", "case", "year")

ds = xr.Dataset(
    {
        "dissolved": (tdim, dissolved.astype(np.float32)),
        "dissolved_diff": (tdim, dissolved_diff.astype(np.float32)),
        "dissolved_pcterr": (tdim, dissolved_pcterr.astype(np.float32)),
        "annual_dissolution": (ydim, annual_dissolution.astype(np.float32)),
        "annual_dissolution_diff": (ydim, annual_dissolution_diff.astype(np.float32)),
        "annual_dissolution_pcterr": (ydim, annual_dissolution_pcterr.astype(np.float32)),
        "inventory0": (("site",),
                       np.array([inventory0[s] * MOLAR_MASS / 1000.0 for s in SITES],
                                dtype=np.float32)),
    },
    coords={
        "site": SITES,
        "case": CASES,
        "time": day_ends,
        "year": years,
        "state": ("site", [states_per_site[s] for s in SITES]),
    },
)

REFTXT = f"the {REFERENCE} case"
ds["dissolved"].attrs = {
    "units": "kg", "long_name": "cumulative forsterite dissolved",
    "description": "M(0) - M(t) of `forst-ph` in the whole domain, which is 1 m2 in plan",
    "cell_methods": "time: point"}
ds["dissolved_diff"].attrs = {
    "units": "kg", "long_name": f"cumulative forsterite dissolved, minus {REFTXT}",
    "description": f"dissolved(case) - dissolved({REFERENCE}); zero by construction at {REFERENCE}"}
ds["dissolved_pcterr"].attrs = {
    "units": "%", "long_name": f"error in cumulative forsterite dissolved, relative to {REFTXT}",
    "description": f"100 x dissolved_diff / dissolved({REFERENCE})",
    "precision_note": ("masked where the hourly cumulative is under ten output quanta of "
                       "`_o.mms` (1e-3 mol, 1.4e-4 kg) -- the first day or so of the run")}
ds["annual_dissolution"].attrs = {
    "units": "kg", "long_name": "forsterite dissolved within the year",
    "description": "the cumulative at this year end minus the cumulative at the previous one",
    "cell_methods": "time: sum (interval: 365 days)"}
ds["annual_dissolution_diff"].attrs = {
    "units": "kg", "long_name": f"forsterite dissolved within the year, minus {REFTXT}",
    "description": f"annual_dissolution(case) - annual_dissolution({REFERENCE})"}
ds["annual_dissolution_pcterr"].attrs = {
    "units": "%",
    "long_name": f"error in forsterite dissolved within the year, relative to {REFTXT}",
    "description": ("100 x annual_dissolution_diff / annual_dissolution(hourly); the error on "
                    "this year's dissolution, not the error on the cumulative")}
ds["inventory0"].attrs = {
    "units": "kg", "long_name": "forsterite present at day 0",
    "description": ("the amendment as applied, per square metre; divide by 0.1 for t/ha. Read it "
                    "rather than assuming 10 t/ha -- these runs are 9.833")}

ds["site"].attrs = {"long_name": "soil series"}
ds["case"].attrs = {
    "long_name": "meteorological forcing resolution",
    "description": ("resolution the 10-year met record was resampled to before it was handed to "
                    f"MIN3P; ordered coarse-ward, {REFERENCE} is the reference")}
# Deliberately *not* CF "days since ..." -- that makes xarray decode this to a datetime64 on
# reopen, and these runs have no anchoring calendar date.
ds["time"].attrs = {"units": "days", "long_name": "end of the day",
                    "description": "elapsed days from the start of the run"}
ds["year"].attrs = {"long_name": "year of the simulation", "description": "1-based; 365 days each"}
ds["state"].attrs = {"long_name": "site location"}

ds.attrs = {
    "title": "MIN3P met_forcing_rxn forsterite dissolution",
    "description": ("Cumulative and per-year forsterite dissolution for each site and met forcing "
                    "resolution, and the error each coarser forcing makes against the "
                    f"{REFERENCE} run."),
    "treatment": f"the `<case>_{TREAT}` runs only; the control runs carry no forsterite",
    "source_variable": f"`{MINERAL}` in `<case>_o.mms`, the domain-integrated mineral inventory",
    "molar_mass_g_per_mol": MOLAR_MASS,
    "domain_plan_area_m2": 1.0,
    "method": ("M(0) - M(t) interpolated from MIN3P's adaptive output times onto day ends; the "
               "annual variables are that same curve sampled at the year ends and differenced"),
    "created_by": "figures/postprocess-data/create-forsterite-dissolution.ipynb",
}

print(dict(ds.sizes))
ds

{'site': 8, 'case': 4, 'time': 3650, 'year': 10}


<xarray.Dataset> Size: 1MB
Dimensions:                    (site: 8, case: 4, time: 3650, year: 10)
Coordinates:
  * site                       (site) <U12 384B 'Cecil' 'Flanagan' ... 'Yolo'
    state                      (site) <U11 352B 'Northern SC' ... 'Central CA'
  * case                       (case) <U8 128B 'hourly' 'daily' ... 'longterm'
  * time                       (time) float64 29kB 1.0 2.0 ... 3.65e+03
  * year                       (year) int64 80B 1 2 3 4 5 6 7 8 9 10
Data variables:
    dissolved                  (site, case, time) float32 467kB 0.0008442 ......
    dissolved_diff             (site, case, time) float32 467kB 0.0 ... 0.05517
    dissolved_pcterr           (site, case, time) float32 467kB nan 0.0 ... 7.43
    annual_dissolution         (site, case, year) float32 1kB 0.2002 ... 0.0408
    annual_dissolution_diff    (site, case, year) float32 1kB 0.0 ... 0.0006613
    annual_dissolution_pcterr  (site, case, year) float32 1kB 0.0 0.0 ... 1.647
    inventory0                 (site) float32 32B 0.9833 0.9833 ... 0.9833
Attributes:
    title:                 MIN3P met_forcing_rxn forsterite dissolution
    description:           Cumulative and per-year forsterite dissolution for...
    treatment:             the `<case>_erw` runs only; the control runs carry...
    source_variable:       `forst-ph` in `<case>_o.mms`, the domain-integrate...
    molar_mass_g_per_mol:  140.6931
    domain_plan_area_m2:   1.0
    method:                M(0) - M(t) interpolated from MIN3P's adaptive out...
    created_by:            figures/postprocess-data/create-forsterite-dissolu...

## 6. Checks

In [11]:
# Nothing dissolves more feedstock than was applied, and nothing runs backwards.
for i, site in enumerate(SITES):
    cap = inventory0[site] * MOLAR_MASS / 1000.0
    assert dissolved[i].max() <= cap + 1e-9, f"{site} dissolved more than was applied"
assert (np.diff(dissolved, axis=-1) >= -1e-12).all(), "cumulative dissolution decreases"

# The two views have to close on each other: the years partition the run, so they sum to the
# cumulative at day 3650.
assert np.allclose(annual_dissolution.sum(-1), dissolved[..., -1], atol=1e-9), \
    "the annual view does not sum to the cumulative"

# The hourly reference is exactly zero in every difference.
for arr in (dissolved_diff, annual_dissolution_diff):
    assert (arr[:, REF] == 0).all()

frac = 100 * dissolved[..., -1] / (np.array([inventory0[s] for s in SITES])[:, None]
                                   * MOLAR_MASS / 1000.0)
print("fraction of the amendment dissolved after 10 years:")
print(pd.DataFrame(frac, index=SITES, columns=CASES).round(1).astype(str) + " %")

print("\nkg dissolved each year, hourly case:")
print(ds["annual_dissolution"].sel(case=REFERENCE).to_pandas().round(4))

print("\n% error on each year's dissolution, mean over sites:")
print(ds["annual_dissolution_pcterr"].mean("site").to_pandas().round(2))

fraction of the amendment dissolved after 10 years:
              hourly   daily monthly longterm
Cecil         94.5 %  94.5 %  95.5 %   96.1 %
Flanagan      94.8 %  94.8 %  96.0 %   96.4 %
HoustonBlack  76.6 %  77.2 %  82.3 %   85.3 %
Kalamazoo     99.0 %  99.0 %  99.7 %   99.9 %
Kuma          57.9 %  58.0 %  59.7 %   65.2 %
Palouse       85.8 %  85.5 %  87.1 %   88.2 %
Pullman       46.1 %  46.0 %  45.1 %   43.5 %
Yolo          75.5 %  75.5 %  79.1 %   81.1 %

kg dissolved each year, hourly case:
year              1       2       3       4       5       6       7       8   \
site                                                                           
Cecil         0.2002  0.1657  0.1344  0.1120  0.0913  0.0729  0.0549  0.0417   
Flanagan      0.1715  0.1465  0.1300  0.1138  0.0969  0.0808  0.0664  0.0527   
HoustonBlack  0.1104  0.1010  0.0922  0.0843  0.0770  0.0690  0.0643  0.0567   
Kalamazoo     0.2651  0.1943  0.1555  0.1194  0.0889  0.0565  0.0368  0.0264   
Kuma          0.

## 7. Write

In [12]:
NAME = "forsterite_dissolution"
url = f"{OUT_ROOT}/{NAME}.zarr"

if WRITE:
    if not OVERWRITE and fs.exists(url.replace("s3://", "")):
        raise FileExistsError(f"{url} exists and OVERWRITE is False")
    # The whole store is a couple of megabytes, so one chunk per array is right.
    ds.chunk({d: n for d, n in ds.sizes.items()}).to_zarr(
        url, mode="w", zarr_format=2, consolidated=True)
    print("wrote", url)
else:
    print(f"WRITE is False -- would write {url}  {dict(ds.sizes)}")

wrote s3://carbonplan-carbon-removal/ew-workflows-data/min3p/simulations/postprocessed/met_forcing_rxn/forsterite_dissolution.zarr


## 8. Read back

In [13]:
if WRITE:
    back = xr.open_zarr(url).load()
    xr.testing.assert_allclose(back, ds)
    mb = sum(f["size"] for f in fs.find(url.replace("s3://", ""), detail=True).values()) / 1e6
    print(f"{NAME:26s} {str(dict(back.sizes)):<52s} {mb:6.3f} MB")
    print("round-trips")

forsterite_dissolution     {'site': 8, 'case': 4, 'year': 10, 'time': 3650}      0.728 MB
round-trips


In [ ]:
# ---